# Phase 0 | DANDI 001132 Feasibility Audit
### Human hippocampal optogenetics: donor independence, NWB contents, optical trials, and labels

**Purpose:** Determine whether DANDI 001132 supports a personalized, uncertainty-aware neural-response predictor and subsequently a simulated closed-loop controller. This notebook **does not train an AI model**.

**Evidence distinction**
1. **Published facts:** Seven epilepsy patients, 12 slices, 8 HcKCR1 slices (5 donors), two ChR2 slices, two slices without an opsin (Andrews et al., 2024 Table 1).
2. **Manifest facts:** Public Dandiset version `0.241130.1903` provides 69 NWB files totaling approximately 192.4 GB.
3. **Not yet proven:** Whether each NWB file includes correctly aligned optical ON/OFF times, dose/intensity, stimulation duration, challenge-condition time windows, and precomputed spike-train/epileptiform labels. The original paper describes a separate CSV stimulation log.

**Important:** Never equate 69 NWB recording IDs with 69 patients. Never treat media-induced epileptiform bursts as clinically annotated seizures.

References:
- DANDI: https://dandiarchive.org/dandiset/001132
- Version-pinned manifest: https://api.dandiarchive.org/api/dandisets/001132/versions/0.241130.1903/assets/
- Paper (Andrews et al. 2024): https://pmc.ncbi.nlm.nih.gov/articles/PMC11881784/
- Experimental source code: https://github.com/braingeneers/AndrewsGengVoitiukEtAl_SourceCode
- DANDI streaming: https://docs.dandiarchive.org/user-guide-using/accessing-data/streaming/

**Kaggle:** Create/upload the notebook, set **Internet = ON**, and run cells top-to-bottom. CPU is sufficient. Default mode *streams metadata* and **does not download entire NWB files**.

## 1. Install small dependencies
Kaggle usually includes `pandas`, `requests`, `h5py`, and `numpy`. Install `remfile` for streaming HDF5 metadata from DANDI.

In [1]:
%pip -q install remfile

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration
`VERSION` is pinned for reproducible science. Start with two small candidate NWB assets (inhibitory slices 5C and 3C). The smallest recording is **not automatically a confirmed stimulation trial**.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import re, json, os, zipfile, traceback
import requests
import numpy as np
import pandas as pd
from IPython.display import display

DANDISET_ID = "001132"
VERSION = "0.241130.1903"
API_ROOT = "https://api.dandiarchive.org/api"
OUT = Path("/kaggle/working/phase0_dandi001132") if Path("/kaggle/working").exists() else Path("phase0_dandi001132")
OUT.mkdir(parents=True, exist_ok=True)

ENABLE_REMOTE_NWB_PROBE = True       # Change to False to run manifest-only audit.
PREFERRED_SLICES = ["5C", "3C", "10F"]
MAX_NWB_PROBES = 2                 # 2 files only to limit remote requests.
REMOTE_ROOT_SCAN_LIMIT = 150       # Metadata objects, NEVER raw signal arrays.

print("Output directory:", OUT.resolve())
print("Pinned DANDI version:", VERSION)
print("Enable Internet in Kaggle settings before running API cells.")

Output directory: /kaggle/working/phase0_dandi001132
Pinned DANDI version: 0.241130.1903
Enable Internet in Kaggle settings before running API cells.


## 3. Published patient/slice reference table
These are **slice-level labels from Table 1** of the original paper, not trial-by-trial labels discovered in the NWB files.

In [3]:
REFERENCE = [
    ("1A", "A", "none", "none"),
    ("2B", "B", "ChR2", "none"),
    ("3C", "C", "HcKCR1", "none"),
    ("4C", "C", "ChR2", "none"),
    ("5C", "C", "HcKCR1", "bicuculline"),
    ("6C", "C", "none", "none"),
    ("7D", "D", "HcKCR1", "bicuculline"),
    ("8D", "D", "HcKCR1", "bicuculline"),
    ("9E", "E", "HcKCR1", "zero magnesium; ± kainic acid"),
    ("10F", "F", "HcKCR1", "zero magnesium; ± kainic acid"),
    ("11G", "G", "HcKCR1", "zero magnesium; ± kainic acid"),
    ("12G", "G", "HcKCR1", "zero magnesium; ± kainic acid"),
]
ref = pd.DataFrame(REFERENCE, columns=["slice_id", "paper_donor_id", "paper_opsin", "paper_media_perturbation"])
assert ref.slice_id.is_unique
assert len(ref) == 12 and ref.paper_donor_id.nunique() == 7
ref.to_csv(OUT / 'reference_published_slices.csv', index=False)
display(ref)

,slice_id,paper_donor_id,paper_opsin,paper_media_perturbation
0,1A,A,none,none
1,2B,B,ChR2,none
2,3C,C,HcKCR1,none
3,4C,C,ChR2,none
4,5C,C,HcKCR1,bicuculline
5,6C,C,none,none
6,7D,D,HcKCR1,bicuculline
7,8D,D,HcKCR1,bicuculline
8,9E,E,HcKCR1,zero magnesium; ± kainic acid
9,10F,F,HcKCR1,zero magnesium; ± kainic acid


## 4. Retrieve the pinned 69-file manifest
Only a compact JSON asset listing is requested; this does **not** download the 192 GB dataset.

Every filename looks like `sub-5C-1/sub-5C-1_ses-..._ecephys.nwb`, where `5C` is **the slice**, `C` is **the donor**, and `1` identifies a recording-level asset.

In [4]:
def fetch_manifest(dandiset_id=DANDISET_ID, version=VERSION):
    endpoint = f"{API_ROOT}/dandisets/{dandiset_id}/versions/{version}/assets/"
    page_url = endpoint
    rows = []
    with requests.Session() as session:
        while page_url:
            response = session.get(page_url, params={"page_size": 100} if page_url == endpoint else None, timeout=45)
            response.raise_for_status()
            data = response.json()
            rows.extend(data["results"])
            page_url = data.get("next")
        return rows

try:
    raw_assets = fetch_manifest()
except Exception as exc:
    raise RuntimeError("Could not contact the DANDI API. Enable Kaggle Internet and rerun this cell.") from exc

pattern = re.compile(r"^sub-(?P<slice>\d+[A-Z])-(?P<recording>\d+)/")
rows = []
for a in raw_assets:
    path = str(a.get("path", ""))
    match = pattern.match(path)
    if not match:
        print("NON-STANDARD PATH (manual review):", path)
        continue
    slice_id = match.group("slice")
    rows.append({
        "asset_id": a["asset_id"], "path": path,
        "file_bytes": int(a["size"]), "file_gb": round(a["size"]/1e9, 3),
        "slice_id": slice_id,
        "donor_id_from_filename": re.search(r"[A-Z]+$", slice_id).group(),
        "recording_index": int(match.group("recording")),
    })
manifest = pd.DataFrame(rows)
assert not manifest.empty
manifest = manifest.merge(ref, how="left", on="slice_id", validate="many_to_one")
assert manifest["asset_id"].is_unique
assert manifest["path"].is_unique

print("NWB files:", len(manifest), "/ archive reports 69")
print("Approximate size (GB):", round(manifest.file_bytes.sum()/1e9, 2))
print("Distinct file-level IDs:", manifest.path.nunique())
print("Distinct slices:", manifest.slice_id.nunique())
print("Distinct donors:", manifest.paper_donor_id.nunique())
print("Missing published slice match:", int(manifest.paper_donor_id.isna().sum()))
manifest.to_csv(OUT / "manifest_69_recordings.csv", index=False)
display(manifest.head(6))

NWB files: 69 / archive reports 69
Approximate size (GB): 192.37
Distinct file-level IDs: 69
Distinct slices: 12
Distinct donors: 7
Missing published slice match: 0


,asset_id,path,file_bytes,file_gb,slice_id,donor_id_from_filename,recording_index,paper_donor_id,paper_opsin,paper_media_perturbation
0,1f3c9771-6abe-4af9-aaac-e7d7606784fa,sub-10F-1/sub-10F-1_ses-20240928T165018_ecephy...,1768928157,1.769,10F,F,1,F,HcKCR1,zero magnesium; ± kainic acid
1,05460e58-ccdc-4da7-8d43-303180f527da,sub-10F-3/sub-10F-3_ses-20240928T165423_ecephy...,2625777228,2.626,10F,F,3,F,HcKCR1,zero magnesium; ± kainic acid
2,1875fa0a-35e1-4b95-bd27-9d56b7328d9f,sub-10F-2/sub-10F-2_ses-20240928T165008_ecephy...,2757405768,2.757,10F,F,2,F,HcKCR1,zero magnesium; ± kainic acid
3,4fde6a2f-1125-427c-a5a3-9cf738abf99e,sub-10F-4/sub-10F-4_ses-20240928T165229_ecephy...,4068167088,4.068,10F,F,4,F,HcKCR1,zero magnesium; ± kainic acid
4,7f4198f9-a0c5-4cd4-a30f-32955e25dcd2,sub-10F-5/sub-10F-5_ses-20240928T165424_ecephy...,4208889800,4.209,10F,F,5,F,HcKCR1,zero magnesium; ± kainic acid
5,80f6f994-c129-4345-b4cf-12dfb63ee1c8,sub-10F-0/sub-10F-0_ses-20240928T165427_ecephy...,3779863088,3.780,10F,F,0,F,HcKCR1,zero magnesium; ± kainic acid


## 5. Independence: files are not samples
A model must never learn from one run of slice 5C and then be tested on a different run of *the same slice* as if the tissue were unseen. Best practice is **leave-one-donor-out** for final assessment and nested validation where feasible.

In [5]:
by_slice = (manifest.groupby(["paper_donor_id","slice_id","paper_opsin","paper_media_perturbation"], dropna=False)
            .agg(n_files=("path","size"), gb=("file_bytes",lambda x:round(x.sum()/1e9,2)))
            .reset_index().sort_values(["paper_donor_id","slice_id"]))
by_donor = (manifest.groupby("paper_donor_id")
            .agg(n_files=("path","size"), n_slices=("slice_id","nunique"))
            .reset_index())
print("Recorded file counts by slice:")
display(by_slice)
print("Donor summary:")
display(by_donor)
print("HcKCR1 inhibitory donors:", sorted(by_slice.query('paper_opsin == "HcKCR1"').paper_donor_id.unique()))
print("HcKCR1 inhibitory slices:", int((by_slice.paper_opsin=="HcKCR1").sum()))
by_slice.to_csv(OUT / "slice_level_manifest_audit.csv", index=False)
by_donor.to_csv(OUT / "donor_level_manifest_audit.csv", index=False)
assert by_slice.n_files.sum() == len(manifest)
assert set(by_slice.slice_id) == set(ref.slice_id)

Recorded file counts by slice:


,paper_donor_id,slice_id,paper_opsin,paper_media_perturbation,n_files,gb
0,A,1A,none,none,1,5.26
1,B,2B,ChR2,none,3,8.01
2,C,3C,HcKCR1,none,8,11.13
3,C,4C,ChR2,none,6,13.28
4,C,5C,HcKCR1,bicuculline,6,17.63
5,C,6C,none,none,5,7.03
6,D,7D,HcKCR1,bicuculline,8,20.30
7,D,8D,HcKCR1,bicuculline,5,14.73
8,E,9E,HcKCR1,zero magnesium; ± kainic acid,9,22.65
9,F,10F,HcKCR1,zero magnesium; ± kainic acid,9,34.17


Donor summary:


,paper_donor_id,n_files,n_slices
0,A,1,1
1,B,3,1
2,C,25,4
3,D,13,2
4,E,9,1
5,F,9,1
6,G,9,2


HcKCR1 inhibitory donors: ['C', 'D', 'E', 'F', 'G']
HcKCR1 inhibitory slices: 8


## 6. What stimulation actions and labels are actually documented?
**Published evidence** is not the same as recoverable machine-learning labels. Do not create artificial dose categories such as 20/40/60/80% unless verified in the raw experiment metadata.

* Optical inhibition: HcKCR1 activated by **530 nm** light; the paper features repeated **10-second continuous illumination** trials, often compared as PRE / LIGHT ON / POST.
* An intensity-sweep analysis is shown for HcKCR1 slices **3C, 5C, and 10F**; a ChR2 intensity sweep is also shown for slice **4C**.
* A threshold-triggered closed-loop experiment was demonstrated in **3 slices from 2 donors**, but the exact machine-readable event count remains unknown.
* Media challenges include physiological medium, **bicuculline**, **zero magnesium**, and **zero magnesium + kainic acid**.
* The original paper describes storing optical light protocol details in a **separate CSV file**. Only NWB files appear in this DANDI manifest.
* The public NWB asset summary names `ElectricalSeries` and `ElectrodeGroup`. It does not by itself prove the presence of `OptogeneticSeries`, trial tables, sorted spike units, or seizure-onset labels.

**Conservative controller action space (only if trial timestamps are verified):** light OFF vs. a fixed 10-s inhibitory-light pulse at *documented* settings. Intensity and temporal-pattern optimization require confirmed multiple observed settings on sufficiently many independent slices.

Sources: Andrews et al., 2024 https://pmc.ncbi.nlm.nih.gov/articles/PMC11881784/ ; DANDI version manifest above.

## 7. Select the smallest **candidate** files, NOT automatically labeled stimulation trials
Default: smallest NWB in slice 5C and 3C (both donor C, for format debugging only). Change `PREFERRED_SLICES` to add 10F, 9E, or 7D after the first run. This notebook does not pull the full raw recording into memory.

In [6]:
candidates = []
for sid in PREFERRED_SLICES:
    subset = manifest[(manifest.slice_id == sid) & (manifest.paper_opsin == "HcKCR1")]
    if subset.empty:
        print("No inhibitory files for", sid)
        continue
    candidates.append(subset.sort_values("file_bytes").iloc[0].to_dict())
selection = pd.DataFrame(candidates).head(MAX_NWB_PROBES)
print("Planned metadata probes (these are candidates, not confirmed light trials):")
display(selection[["slice_id","paper_donor_id","file_gb","path","asset_id"]])
selection.to_csv(OUT / "remote_probe_candidates.csv", index=False)

Planned metadata probes (these are candidates, not confirmed light trials):


,slice_id,paper_donor_id,file_gb,path,asset_id
0,5C,C,0.572,sub-5C-1/sub-5C-1_ses-20240928T165106_ecephys.nwb,5134b6e4-1ce1-4314-ae99-677b6c864840
1,3C,C,0.751,sub-3C-2/sub-3C-2_ses-20240928T192008_ecephys.nwb,495aa69c-4ee8-4553-abfa-298a23394466


## 8. Stream only NWB/HDF5 structure (no raw-array reads)
**What this can establish:** Which groups, metadata fields, and tables exist; whether the file includes `/stimulus`, `/intervals`, `/units`; whether there are optical trial/event candidates. 

**What it cannot establish:** Precise optical event counts or calibrated dose merely by seeing a name with `stim` in it. Those require direct inspection of actual event arrays and any separate CSV logs. The notebook preserves uncertainty rather than inventing labels.

If networking or remote access fails, the error is recorded. This is not evidence that a biological field is missing.

In [7]:
import h5py
import remfile

STIM_WORDS = re.compile(r"opto|optical|photo|stim|led|light|laser|pulse|intensity|power|event|trial|drug|kain|bicuc", re.I)
OPTICAL_WORDS = re.compile(r"opto|optical|photo|led|light|laser|irradiance", re.I)

def get_remote_url(asset_id):
    endpoint = f"{API_ROOT}/assets/{asset_id}/download/"
    r = requests.get(endpoint, stream=True, timeout=60, allow_redirects=True)
    try:
        r.raise_for_status()
        if r.url == endpoint:
            raise ValueError("Asset did not redirect to actual NWB bytes")
        return r.url
    finally:
        r.close()

def decode_scalar(x):
    if isinstance(x, bytes):
        return x.decode('utf-8','replace')
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        if x.size == 1:
            return decode_scalar(x.reshape(-1)[0])
        return f"array of shape {x.shape}"
    return str(x) if not isinstance(x, (str,int,float,bool)) else x

def summarize_structure(h5, scan_limit=REMOTE_ROOT_SCAN_LIMIT):
    records=[]
    branch_roots = ["/acquisition", "/stimulus", "/intervals", "/processing", "/general", "/units"]
    keys = list(h5.keys())

    def visit(obj, path, depth):
        if len(records) >= scan_limit: return
        if isinstance(obj,h5py.Dataset):
            shape = tuple(obj.shape)
            typ = "Dataset"
        else:
            shape = None
            typ = "Group"
        rec={"path":path,"type":typ,"shape":shape}
        if isinstance(obj,h5py.Group):
            ntype=obj.attrs.get('neurodata_type', None)
            rec['neurodata_type']=str(decode_scalar(ntype)) if ntype is not None else ''
        records.append(rec)
        if depth>=2 or not isinstance(obj,h5py.Group): return
        for name in list(obj.keys())[:45]:
            if len(records)>=scan_limit: break
            try:
                visit(obj[name],path.rstrip('/')+'/'+name,depth+1)
            except Exception as e:
                records.append({'path':path+'/'+name,'type':'unreadable','error':str(e)[:180]})

    for path in branch_roots:
        if path in h5:
            visit(h5[path],path,0)

    candidates = [r for r in records if STIM_WORDS.search(r.get('path','')) or STIM_WORDS.search(r.get('neurodata_type',''))]
    interval_info=[]
    if '/intervals' in h5 and isinstance(h5['/intervals'],h5py.Group):
        for name, node in h5['/intervals'].items():
            if not isinstance(node,h5py.Group): continue
            cols=list(node.keys())[:60]
            n = (int(node['start_time'].shape[0]) if 'start_time' in node and isinstance(node['start_time'],h5py.Dataset) else None)
            is_optical = bool(OPTICAL_WORDS.search(name)) or bool(OPTICAL_WORDS.search(' '.join(cols)))
            interval_info.append({'table':name,'n_intervals':n,'columns':cols,'possible_optical_annotation':is_optical})

    # Only the existence of a reliably identified, timestamped optical series/table
    # would provide an optical-event *candidate*, NOT a verified usable action label.
    optic_group_paths=[r['path'] for r in candidates if re.search(r'opto|optical|photo|led|light|laser',r['path']+' '+r.get('neurodata_type',''), re.I)]
    candidate_interval_tables=[x for x in interval_info if x['possible_optical_annotation']]
    subject_id = None
    if '/general/subject/subject_id' in h5:
        ds=h5['/general/subject/subject_id']
        if isinstance(ds,h5py.Dataset) and ds.shape == ():
            subject_id=decode_scalar(ds[()])
    return {
        'top_level_groups':keys, 'subject_id_in_nwb':subject_id,
        'has_units_group':'/units' in h5,
        'has_precomputed_spike_times':'/units/spike_times' in h5,
        'has_stimulus_group':'/stimulus' in h5,
        'has_interval_group':'/intervals' in h5,
        'has_general_optogenetics':'/general/optogenetics' in h5,
        'optical_named_paths':optic_group_paths[:40],
        'interval_tables':interval_info,
        'candidate_optical_interval_tables':candidate_interval_tables,
        'possible_stimulation_metadata_paths':[x.get('path') for x in candidates[:60]],
        'metadata_tree':records,
    }

def probe_remote_asset(record):
    info = {
        'path': record['path'], 'asset_id':record['asset_id'],
        'slice_id':record['slice_id'], 'donor':record['paper_donor_id'],
        'status':'pending', 'error':None
    }
    stream = None
    try:
        url=get_remote_url(record['asset_id'])
        stream=remfile.File(url=url)
        with h5py.File(stream, mode='r') as h5:
            info.update(summarize_structure(h5))
        info['status']='opened_metadata_only'
    except Exception as e:
        info['status']='could_not_open'; info['error']=repr(e)[:700]
    finally:
        if stream is not None:
            try: stream.close()
            except Exception: pass
    return info

if ENABLE_REMOTE_NWB_PROBE:
    reports=[]
    for item in selection.to_dict(orient='records'):
        print('Inspecting NWB structure:', item['path'])
        rep=probe_remote_asset(item)
        reports.append(rep)
        print('Status:', rep['status'])
        if rep['error']: print('Issue:', rep['error'])
else:
    reports=[]
    print('Remote probing disabled; no trial metadata has been checked.')

(OUT/'nwb_probe_full.json').write_text(json.dumps(reports,indent=2,default=str))
flat=[]
for r in reports:
    flat.append({
        'path':r['path'],'slice_id':r['slice_id'],'donor':r['donor'],
        'status':r['status'],'has_units_group':r.get('has_units_group'),
        'has_precomputed_spike_times':r.get('has_precomputed_spike_times'),
        'has_stimulus_group':r.get('has_stimulus_group'),
        'has_interval_group':r.get('has_interval_group'),
        'has_general_optogenetics':r.get('has_general_optogenetics'),
        'optical_named_paths':' | '.join(r.get('optical_named_paths',[])),
        'interval_tables':json.dumps(r.get('interval_tables',[])),
        'error':r['error']
    })
probe_table=pd.DataFrame(flat)
probe_table.to_csv(OUT/'nwb_probe_summary.csv',index=False)
display(probe_table)

Inspecting NWB structure: sub-5C-1/sub-5C-1_ses-20240928T165106_ecephys.nwb
Status: opened_metadata_only
Inspecting NWB structure: sub-3C-2/sub-3C-2_ses-20240928T192008_ecephys.nwb
Status: opened_metadata_only


,path,slice_id,donor,status,has_units_group,has_precomputed_spike_times,has_stimulus_group,has_interval_group,has_general_optogenetics,optical_named_paths,interval_tables,error
0,sub-5C-1/sub-5C-1_ses-20240928T165106_ecephys.nwb,5C,C,opened_metadata_only,False,False,True,False,False,,[],None
1,sub-3C-2/sub-3C-2_ses-20240928T192008_ecephys.nwb,3C,C,opened_metadata_only,False,False,True,False,False,,[],None


## 9. Honest assessment of labels and action identifiability
Interpret NWB structure conservatively:

| Evidence | Meaning |
|---|---|
| `/acquisition/ElectricalSeries` or equivalent | Raw neural voltage likely available; spike sorting or threshold detection may still be needed. |
| `/units/spike_times` | Sorted spike times may already be available. |
| `/stimulus` or `/general/optogenetics` | Stimulation-related metadata may be stored, but action events **are not automatically confirmed**. |
| `/intervals/*/start_time` and `stop_time` | Some time intervals are present; **not automatically optical trials**. |
| Verified LED ON/OFF timestamps with matching electrophysiology clock | Supports learning a fixed ON/OFF response. |
| Calibrated power/intensity per trial | Needed for continuous-dose modeling. |
| Media onset and condition windows | Needed to label spontaneous vs induced hyperactivity. |
| Clinician-adjudicated seizure onset labels | **Not expected**: these are ex-vivo slices, not clinical EEG monitoring. |

The originally described *external CSV logs* may need to be located in the authors' source repository or requested from the corresponding authors, if NWB lacks them.

**Do not use filename recording indices (`-0`, `-1`) as condition labels or stimulation labels.**

In [8]:
requested_fields=[
    ('Patient/slice grouping','YES','Paper Table 1 and file names'),
    ('Opsin type per slice','YES','Paper Table 1'),
    ('Published media challenges per slice','PARTIAL','Table 1, NOT the timing of challenge transitions'),
    ('Optical ON/OFF timestamps per recording','NEEDS_NWB_CHECK','Not proven by manifest or paper'),
    ('Intensity / calibrated power per trial','NEEDS_LOG_CHECK','Original paper reports separate CSV light-protocol logs'),
    ('Multiple distinct intensities in enough donors','UNLIKELY_FOR_MAIN_COHORT','Only limited published intensity-sweep examples'),
    ('Sorted spikes as NWB unit tables','NEEDS_NWB_CHECK','Archive summary lists ElectricalSeries, not a guarantee of units'),
    ('Clinical seizure-onset labels','NO','Ex-vivo induced neural activity is not patient seizure annotation'),
]
audit=pd.DataFrame(requested_fields,columns=['field','status','reason'])
audit.to_csv(OUT/'label_and_action_feasibility.csv',index=False)
display(audit)

opened=[r for r in reports if r.get('status')=='opened_metadata_only']
print('Successfully inspected NWB structures:',len(opened),'of',len(reports))
if not opened:
    print('VERDICT: PENDING NWB verification. Manifest-level and publication-level audit complete.')
else:
    any_meta=any(r.get('optical_named_paths') or r.get('candidate_optical_interval_tables') for r in opened)
    if any_meta:
        print('VERDICT: Optical-related metadata candidates found; verify clocks, actual pulse values and logs next.')
    else:
        print('VERDICT: No obvious optical tables found in the limited scan. External CSVs may be essential.')
print('INTERVENTION ML DOES NOT PASS the readiness gate until trial timestamps and actions are validated.')

,field,status,reason
0,Patient/slice grouping,YES,Paper Table 1 and file names
1,Opsin type per slice,YES,Paper Table 1
2,Published media challenges per slice,PARTIAL,"Table 1, NOT the timing of challenge transitions"
3,Optical ON/OFF timestamps per recording,NEEDS_NWB_CHECK,Not proven by manifest or paper
4,Intensity / calibrated power per trial,NEEDS_LOG_CHECK,Original paper reports separate CSV light-prot...
5,Multiple distinct intensities in enough donors,UNLIKELY_FOR_MAIN_COHORT,Only limited published intensity-sweep examples
6,Sorted spikes as NWB unit tables,NEEDS_NWB_CHECK,"Archive summary lists ElectricalSeries, not a ..."
7,Clinical seizure-onset labels,NO,Ex-vivo induced neural activity is not patient...


Successfully inspected NWB structures: 2 of 2
VERDICT: No obvious optical tables found in the limited scan. External CSVs may be essential.
INTERVENTION ML DOES NOT PASS the readiness gate until trial timestamps and actions are validated.


## 10. Exports: one ZIP for discussing results
After Kaggle finishes, download the ZIP from **Output → `/kaggle/working/phase0_dandi001132_outputs.zip`** and share it in the next research discussion. It includes the live manifest, donor/slice counts, and the NWB metadata check results.

In [9]:
summary = {
    'dandiset':DANDISET_ID,'version':VERSION,
    'run_utc':datetime.now(timezone.utc).isoformat(),
    'manifest_nwb_count':len(manifest),
    'manifest_size_gb':round(manifest.file_bytes.sum()/1e9,3),
    'unique_donors':int(manifest.paper_donor_id.nunique()),
    'unique_slices':int(manifest.slice_id.nunique()),
    'inhibitory_opsin_slices':int((by_slice.paper_opsin=='HcKCR1').sum()),
    'inhibitory_opsin_donors':int(by_slice.query('paper_opsin=="HcKCR1"').paper_donor_id.nunique()),
    'nwb_probe_opened':len(opened),
    'optical_trials_fully_verified':False,
    'optical_intensity_fully_verified':False,
    'notes':'Trial and dose counts deliberately NOT fabricated from filename or mere group existence.'
}
(OUT/'phase0_summary.json').write_text(json.dumps(summary,indent=2))
zip_path=OUT.parent/'phase0_dandi001132_outputs.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob('*')):
        if p.is_file(): z.write(p,arcname=p.name)
print('Saved ZIP:', zip_path)
print(json.dumps(summary,indent=2))

Saved ZIP: /kaggle/working/phase0_dandi001132_outputs.zip
{
  "dandiset": "001132",
  "version": "0.241130.1903",
  "run_utc": "2026-10-09T21:20:17.517035+00:00",
  "manifest_nwb_count": 69,
  "manifest_size_gb": 192.373,
  "unique_donors": 7,
  "unique_slices": 12,
  "inhibitory_opsin_slices": 8,
  "inhibitory_opsin_donors": 5,
  "nwb_probe_opened": 2,
  "optical_trials_fully_verified": false,
  "optical_intensity_fully_verified": false,
  "notes": "Trial and dose counts deliberately NOT fabricated from filename or mere group existence."
}


## 11. Go / No-Go decision for our proposed Q1 project

**Route A - response prediction (recommended first):** If synchronized optical event on/off markers are recoverable, measure PRE / ON / POST changes and train baseline response prediction using **HcKCR1 samples**. Primary end points: firing rate reduction, burst statistics and uncertainty calibration. Evaluate using donor-disjoint splits. This is credible even when multiple light doses are not recoverable.

**Route B - restricted simulation of control (conditional):** If we can identify repeated episodes with actions on/off, build a model of a *fixed 10-second pulse*, then simulate a threshold and a conservative model-predictive trigger policy. Report these as **in-silico outcomes, not real patient intervention**. Test held-out stimuli and alternative simulators to identify model exploitation.

**Route C - broader dose optimization (not currently supported):** Do not optimize arbitrary light intensity, duration, pulse frequency or precise energy until independent subjects, trial-level action metadata and time alignment are confirmed.

**Critical missing materials to pursue:** signed/local raw HDF5 acquisition plus per-recording optical logs/calibration CSV files; a recording ↔ donor ↔ slice ↔ medium ↔ illumination table; stimulus clocks/UTC offsets; exact number of repeats and exposure conditions. See the original research source repository for code and contact the corresponding authors for unavailable metadata.

**Next notebook, only after this audit:** `01_NWB_Electrophysiology_and_Stimulus_Alignment.ipynb` - inspect one slice, locate true optical timestamps, align with raw voltage or sorted spikes and create one honest PRE / LIGHT ON / POST figure.